# 03 — Feature engineering

Build leakage-safe predictors and 24-hour target vectors for each station in the wide joined partitions.

**Inputs:** `data/processed/joined/all_stations_train.parquet`, `all_stations_test.parquet`  
**Outputs:** `data/processed/joined/all_stations_train_features.parquet`, `all_stations_test_features.parquet`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import STATION_IDS, WEATHER_VARIABLES
from src.feature_engineering import (
    DEFAULT_FEATURE_CONFIG,
    build_feature_frame,
    feature_column_names,
    target_column_names,
)

JOINED_DIR = Path("data/processed/joined")
PARTITIONS = ("train", "test")
SOURCE_COLUMNS = ("water_level", "imputed", "station_id", *WEATHER_VARIABLES)
DERIVED_FEATURE_COLUMNS = tuple(
    column
    for column in feature_column_names(DEFAULT_FEATURE_CONFIG)
    if column not in SOURCE_COLUMNS
)
DERIVED_COLUMNS = (
    *DERIVED_FEATURE_COLUMNS,
    "target_valid",
    *target_column_names(DEFAULT_FEATURE_CONFIG),
)
OUTPUT_PATHS = {
    partition: JOINED_DIR / f"all_stations_{partition}_features.parquet"
    for partition in PARTITIONS
}

## Leakage contract

The wide train and sealed-test joined Parquets are read independently. Each station's prefixed source columns are extracted into a single-station frame before calling `build_feature_frame`; partitions and stations are never combined for feature calculation, so the beginning of the test partition cannot inherit train lookbacks.

Every predictor uses information available through issue time $t$: positive shifts create lags, and trailing windows include $t$, require a complete window, and propagate source missingness. Future weather is never used.

Targets are `water_level` at $t+1$ through $t+24$. `target_valid` is calculated independently inside each station frame and requires the full future target window to be observed and non-imputed; otherwise all 24 target values are null. Original joined columns and the shared timestamp are preserved unchanged. Only derived predictors, `target_valid`, and targets are added under each station prefix.

No rows are dropped from the joined outputs, no missing values are filled, no scaling is applied, and no `features_valid` shortcut is introduced. Rows where a station is unavailable remain null in its derived region, including stations with no usable rows in a partition.

In [ ]:
def extract_station_frame(
    joined: pd.DataFrame, station_id: str
) -> tuple[pd.DataFrame, pd.Series]:
    """Extract the contiguous usable rows for one prefixed station."""
    prefix = f"{station_id}__"
    prefixed_columns = [f"{prefix}{column}" for column in SOURCE_COLUMNS]
    missing = sorted(set(prefixed_columns).difference(joined.columns))
    if missing:
        raise ValueError(f"{station_id} is missing joined columns: {missing}")

    available = joined[f"{prefix}station_id"].notna()
    station = joined.loc[available, ["timestamp", *prefixed_columns]].rename(
        columns={f"{prefix}{column}": column for column in SOURCE_COLUMNS}
    )
    station = station.reset_index(drop=True)
    if not station.empty:
        # Joined null regions make this column object-typed; usable rows are Boolean.
        station["imputed"] = station["imputed"].astype(bool)
    return station, available


def attach_station_features(
    joined: pd.DataFrame,
    station_id: str,
    available: pd.Series,
    station_features: pd.DataFrame,
) -> pd.DataFrame:
    """Add one station's derived columns while preserving the joined frame."""
    result = joined.copy()
    values_index = joined.index[available]
    prefix = f"{station_id}__"
    for column in DERIVED_COLUMNS:
        if column == "target_valid":
            values = pd.Series(pd.NA, index=joined.index, dtype="boolean")
        else:
            values = pd.Series(float("nan"), index=joined.index, dtype="float64")
        if not station_features.empty:
            values.loc[values_index] = station_features[column].to_numpy()
        result[f"{prefix}{column}"] = values
    return result


def build_joined_feature_frame(
    joined: pd.DataFrame, partition: str
) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    """Engineer each station in one joined partition independently."""
    result = joined.copy()
    partition_summaries: list[dict[str, object]] = []
    for station_id in STATION_IDS:
        station, available = extract_station_frame(joined, station_id)
        if station.empty:
            result = attach_station_features(
                result, station_id, available, station_features=station
            )
            target_valid_rows = 0
        else:
            station_features = build_feature_frame(
                station, station_id=station_id, config=DEFAULT_FEATURE_CONFIG
            )
            result = attach_station_features(
                result, station_id, available, station_features
            )
            target_valid_rows = int(station_features["target_valid"].sum())
        partition_summaries.append(
            {
                "partition": partition,
                "station_id": station_id,
                "usable_rows": int(available.sum()),
                "target_valid_rows": target_valid_rows,
            }
        )
    return result, partition_summaries


summaries: list[dict[str, object]] = []
for partition in PARTITIONS:
    source_path = JOINED_DIR / f"all_stations_{partition}.parquet"
    joined_source = pd.read_parquet(source_path)
    joined_features, partition_summaries = build_joined_feature_frame(
        joined_source, partition
    )
    if (
        joined_features.columns[: len(joined_source.columns)].tolist()
        != joined_source.columns.tolist()
    ):
        raise AssertionError(f"{partition} source columns were not preserved")
    if not joined_features["timestamp"].equals(joined_source["timestamp"]):
        raise AssertionError(f"{partition} timestamps were not preserved")
    joined_features.to_parquet(OUTPUT_PATHS[partition], index=False)
    for summary in partition_summaries:
        summaries.append(
            {
                **summary,
                "rows": len(joined_features),
                "columns": len(joined_features.columns),
                "path": str(OUTPUT_PATHS[partition]),
            }
        )

In [ ]:
artifact_summary = pd.DataFrame(summaries)
contract_summary = pd.DataFrame(
    {
        "contract": [
            "original joined columns",
            "derived predictors per station",
            "targets per station",
            "forecast horizon (hours)",
        ],
        "count": [
            len(joined_source.columns),
            len(DERIVED_FEATURE_COLUMNS),
            len(target_column_names(DEFAULT_FEATURE_CONFIG)),
            DEFAULT_FEATURE_CONFIG.horizon_hours,
        ],
    }
)
display(artifact_summary)
display(contract_summary)

Wide joined feature outputs were regenerated successfully. Source columns and timestamps are preserved, each usable station was engineered independently in each partition, and unavailable station regions retain null derived values.